<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/main/src/eval/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Libraries

In [1]:
!pip install -q transformers>=4.45.0 accelerate torch torchvision pillow scikit-learn tqdm chess cairosvg python-Levenshtein

Importing Libraries

In [2]:
import os
import sys
import json
import subprocess
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import AutoModelForImageTextToText, AutoProcessor

Setting up environment

In [3]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent

# 2. Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

print(f"Setup Complete. REPO_ROOT: {repo_root}")

# 3. Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 4. Import custom project modules cleanly (from data)
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

# 5. Import evaluation modules/utilities from src/eval/utilities.py
from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_1,
)

print("All custom modules and eval utilities imported successfully!")

Repository directory already exists at: /content/BigDataAndTextMiningProject
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: Tesla T4
All custom modules and eval utilities imported successfully!


Dowloading dataset from HuggingFace repo

In [4]:
print("Verifying Authentication to Hugging Face...")
notebook_login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

# Specifichiamo esplicitamente 'imagefolder' per dire a Hugging Face
# di leggere la struttura basata su metadata.jsonl che abbiamo generato
dataset_task1 = load_dataset(dataset_name)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Verifying Authentication to Hugging Face...


Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': '2GDeK', 'task': 'task1', 'fen': 'r1b2rk1/ppRq3p/3p2p1/3PPp2/8/3B1NP1/2Q2K1P/8 b - - 1 30', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of all pieces on th

Loading Baseline Model

In [5]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model Qwen/Qwen3.5-0.8B...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model and Processor loaded correctly!


Test with baseline

In [6]:
# 1. Grab the first test sample directly from your loaded Hugging Face dataset variable
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]

# 2. Use the image already downloaded and loaded by Hugging Face dataset
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# 3. Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# 4. Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# 5. Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model.device)

# 6. Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model.generate(**model_inputs, max_new_tokens=128)

# 7. Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Sample ID: sample_000000
FEN: 4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - - 5 28
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - - 5 28

Generating zero-shot prediction...
Predicted FEN (Zero-Shot): a1b2c3d4e5f6g7h8
8h7g7e8
7d6c5b4a3
4e3d2c1b0
1f2g1h0
0d0e0f0g0h0
0a0b0c0d0e0f0g0h0
0a0b0c0d0e0f0g0h0


In [7]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_1(
    model=model,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

Evaluating Vanilla Qwen2.5-VL (Zero-Shot): 100%|██████████| 4/4 [01:04<00:00, 16.10s/it]


Evaluation completed for Vanilla Qwen2.5-VL (Zero-Shot)! Results saved to task1_vanilla_qwen2.5-vl_(zero-shot)_results.csv.

Example of results obtained form the evaluation:


,sample_id,ground_truth,predicted,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy
0,sample_000000,4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - ...,a1b2c3d4e5f6g7h8\n8h7g7e8\n7d6c5b4a3\n4e3d2c1b...,0.0,92,1.769231,0.0
1,sample_000001,3q1rk1/1p1bbppp/p3p3/1n1pP3/3N1P2/3QB2P/PPB3P1...,a1b2b3b4b5b6b7b8c8d8d9d9e9e1e1f1f2f3f4f5f6f7g7...,0.0,58,0.920635,0.0
2,sample_000002,8/2Kbk3/1B1p4/2pPp3/2B1Pp2/pP6/Pr3R2/8 w - - 1 51,a2 b2 c2 d2 e2 f2 g2 h2\na3 b3 c3 d3 e3 f3 g3 ...,0.0,133,2.714286,0.0
3,sample_000003,6k1/4bppp/2b1p3/1pNpP3/3P4/P3B3/r4PPP/2R3K1 w ...,a1b2b3b4b5b6b7b8c8d8e8f8g8h8,0.0,48,0.888889,0.0



Comparative Summary DataFrame (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.57326,0.0,82.75


In [8]:
display(vanilla_results_df.head())

,sample_id,ground_truth,predicted,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy
0,sample_000000,4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - ...,a1b2c3d4e5f6g7h8\n8h7g7e8\n7d6c5b4a3\n4e3d2c1b...,0.0,92,1.769231,0.0
1,sample_000001,3q1rk1/1p1bbppp/p3p3/1n1pP3/3N1P2/3QB2P/PPB3P1...,a1b2b3b4b5b6b7b8c8d8d9d9e9e1e1f1f2f3f4f5f6f7g7...,0.0,58,0.920635,0.0
2,sample_000002,8/2Kbk3/1B1p4/2pPp3/2B1Pp2/pP6/Pr3R2/8 w - - 1 51,a2 b2 c2 d2 e2 f2 g2 h2\na3 b3 c3 d3 e3 f3 g3 ...,0.0,133,2.714286,0.0
3,sample_000003,6k1/4bppp/2b1p3/1pNpP3/3P4/P3B3/r4PPP/2R3K1 w ...,a1b2b3b4b5b6b7b8c8d8e8f8g8h8,0.0,48,0.888889,0.0


Vanilla model is done, let's start finetuning it with LoRA

In [9]:
!pip install --upgrade torchao

Questa funzione va poi messa negli utilities.py

In [ ]:
def preprocess_function(sample):
    chat_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": sample["target"]},
            ],
        }
    ]

    text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=False)

    batch = processor(
        text=[text],
        images=[sample["image"]],
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    batch = {k: v[0] for k, v in batch.items()}
    batch["labels"] = batch["input_ids"].clone()
    batch["labels"][batch["labels"] == processor.tokenizer.pad_token_id] = -100

    return batch

In [10]:
import torch
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer

# 1. Unified preprocessing function for Qwen-VL


print("Applying preprocessing to datasets...")
tokenized_train = dataset_task1["train"].map(preprocess_function, remove_columns=dataset_task1["train"].column_names)
tokenized_val = dataset_task1["validation"].map(preprocess_function, remove_columns=dataset_task1["validation"].column_names)

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to 'model' but save the resulting PEFT model into a NEW variable: 'lora_model'
lora_model = get_peft_model(model, peft_config)
lora_model.print_trainable_parameters()

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen_task1_lora_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    remove_unused_columns=False,
    report_to="none"
)

# 4. Initialize the Trainer using 'lora_model'
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# 5. Start Fine-Tuning
print("Starting LoRA Supervised Fine-Tuning...")
trainer.train()

Applying preprocessing to datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435
Starting LoRA Supervised Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,No log,15.430305
2,No log,14.748506


Fine-tuning completed successfully and LoRA weights saved!

Evaluating the LoRA fine-tuned model on the test set...


NameError: name 'evaluate_chessboard_model' is not defined

In [11]:
# 6. Save weights to a dedicated folder
trainer.model.save_pretrained("./qwen_task1_lora_final")
processor.save_pretrained("./qwen_task1_lora_final")
print("Fine-tuning completed successfully and LoRA weights saved!")

# 7. Evaluate the new 'lora_model' and update the comparative summary table
print("\nEvaluating the LoRA fine-tuned model on the test set...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_1(
    model=lora_model,          # <-- Usiamo la nuova variabile
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Qwen + LoRA Fine-Tuning"
)

# Append the new metrics to the global comparison DataFrame without touching the vanilla 'model'
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)

Fine-tuning completed successfully and LoRA weights saved!

Evaluating the LoRA fine-tuned model on the test set...


Evaluating Qwen + LoRA Fine-Tuning: 100%|██████████| 4/4 [01:08<00:00, 17.02s/it]


Evaluation completed for Qwen + LoRA Fine-Tuning! Results saved to task1_qwen_+_lora_fine-tuning_results.csv.

Updated Comparative Summary Table (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.573260,0.0,82.75
1,Qwen + LoRA Fine-Tuning,0.0,1.361722,0.0,71.75


RICOMINCIARE DA QUA CON GLI ALTRI MODELLI CAMBIANO L'ORDINE DELLE PATCH